# 02 — Leak-free data processing

Mục tiêu của notebook là chứng minh data contract, split contract và preprocessing parity. Notebook **không chọn model bằng frozen test**. Mọi imputer/encoder/scaler được fit trong pipeline trên training folds.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate

from telco_churn.data import (
    create_or_load_split_manifest,
    load_modeling_data,
    split_by_manifest,
    split_manifest_metadata,
)
from telco_churn.features import build_estimator, make_preprocessor
from telco_churn.schema import CATEGORICAL_FEATURES, FEATURE_COLUMNS, SPLIT_SEED

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
DATA_PATH = ROOT / "data/raw/Telco-Customer-Churn.csv"
SPLIT_PATH = ROOT / "data/splits/split_manifest.csv"

## 1. Modeling contract

19 features được phép tại prediction point, một ID chỉ để join/tie-break, một binary target. Không tạo feature từ toàn bộ dataset trước split; không dùng `customerID`; không dùng label-derived aggregates.

In [2]:
features, target, customer_ids, quality = load_modeling_data(DATA_PATH)
contract = pd.DataFrame({
    "column": FEATURE_COLUMNS,
    "role": ["categorical" if column in CATEGORICAL_FEATURES else "numeric" for column in FEATURE_COLUMNS],
    "missing": [int(features[column].isna().sum()) for column in FEATURE_COLUMNS],
    "cardinality": [features[column].nunique(dropna=False) for column in FEATURE_COLUMNS],
})
display(contract)
assert features.columns.tolist() == FEATURE_COLUMNS
assert not features.columns.isin(["customerID", "Churn", "churn_flag"]).any()

,column,role,missing,cardinality
0,gender,categorical,0,2
1,SeniorCitizen,categorical,0,2
2,Partner,categorical,0,2
3,Dependents,categorical,0,2
4,PhoneService,categorical,0,2
5,MultipleLines,categorical,0,3
6,InternetService,categorical,0,3
7,OnlineSecurity,categorical,0,3
8,OnlineBackup,categorical,0,3
9,DeviceProtection,categorical,0,3


## 2. Frozen split manifest

Manifest được tạo đúng một lần bằng stratified seed `20260913`, lưu ID → split → target-at-split và hash. Lần chạy sau phải tái sử dụng; code sẽ fail nếu ID, label hay seed thay đổi. Vì dataset không có timestamp, đây chỉ là portfolio proxy cho out-of-time split.

In [3]:
manifest = create_or_load_split_manifest(customer_ids, target, SPLIT_PATH, seed=SPLIT_SEED)
x_dev, x_test, y_dev, y_test, ids_dev, ids_test = split_by_manifest(
    features, target, customer_ids, manifest
)
display(pd.DataFrame({
    "rows": [len(x_dev), len(x_test)],
    "churn_rate": [y_dev.mean(), y_test.mean()]
}, index=["development", "frozen_test"]))
display(pd.Series(split_manifest_metadata(SPLIT_PATH), name="value").to_frame())
assert set(ids_dev).isdisjoint(set(ids_test))
assert len(x_dev) + len(x_test) == 7_043

,rows,churn_rate
development,5634,0.265353
frozen_test,1409,0.265436


,value
path,D:/GitHub/machine/data/splits/split_manifest.csv
sha256,08a34a4e87b8fbf143299650fad0fecb459552336db345...
rows,7043
development_rows,5634
test_rows,1409


## 3. Preprocessing graph

- Numeric: `TotalCharges` missing → 0, sau đó standardize.
- Categorical: most-frequent imputation (defensive) → one-hot, `handle_unknown='ignore'`.
- Logistic dùng sparse matrix; HistGradientBoosting dùng dense OHE; CatBoost nhận categorical columns trực tiếp.
- Tất cả transform nằm trong estimator/Pipeline nên được refit riêng cho từng fold.

In [4]:
preprocessor = make_preprocessor(dense=False)
transformed = preprocessor.fit_transform(x_dev)
feature_names = preprocessor.get_feature_names_out()
print({"rows": transformed.shape[0], "output_features": transformed.shape[1], "sparse": hasattr(transformed, "nnz")})
display(pd.Series(feature_names, name="transformed_feature").head(30).to_frame())
assert transformed.shape[0] == len(x_dev)
assert transformed.shape[1] == len(feature_names)

{'rows': 5634, 'output_features': 46, 'sparse': False}


,transformed_feature
0,numeric__tenure
1,numeric__MonthlyCharges
2,numeric__TotalCharges
3,categorical__gender_Female
4,categorical__gender_Male
5,categorical__SeniorCitizen_0
6,categorical__SeniorCitizen_1
7,categorical__Partner_No
8,categorical__Partner_Yes
9,categorical__Dependents_No


Cell trên chỉ minh họa transform. Trong benchmark thật, `RandomizedSearchCV` nhận nguyên Pipeline; không tái sử dụng preprocessor đã fit ở đây.

## 4. CV smoke test trên development — không chạm test

In [5]:
baseline = build_estimator("logistic", seed=SPLIT_SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SPLIT_SEED)
scores = cross_validate(
    baseline, x_dev, y_dev, cv=cv,
    scoring={"ap": "average_precision", "roc_auc": "roc_auc", "brier": "neg_brier_score"},
    n_jobs=-1, return_train_score=False,
)
display(pd.DataFrame({
    "metric": ["Average Precision", "ROC-AUC", "Brier (lower better)"],
    "mean": [scores["test_ap"].mean(), scores["test_roc_auc"].mean(), -scores["test_brier"].mean()],
    "std": [scores["test_ap"].std(ddof=1), scores["test_roc_auc"].std(ddof=1), (-scores["test_brier"]).std(ddof=1)],
}))

,metric,mean,std
0,Average Precision,0.661797,0.037782
1,ROC-AUC,0.844182,0.017426
2,Brier (lower better),0.135819,0.007125


## 5. Unknown-category và missing-data robustness

In [6]:
fitted = clone(baseline).fit(x_dev, y_dev)
synthetic = x_dev.iloc[[0]].copy()
synthetic.loc[:, "PaymentMethod"] = "Future payment rail"
synthetic.loc[:, "TotalCharges"] = pd.NA
probability = fitted.predict_proba(synthetic)[:, 1]
print({"unknown_category_probability": float(probability[0])})
assert 0 <= probability[0] <= 1

{'unknown_category_probability': 0.6212147637525411}


`handle_unknown='ignore'` giữ service chạy được, nhưng production vẫn phải log unknown-category rate để phát hiện drift. API strict enum cố ý reject giá trị lạ (HTTP 422); batch pipeline có thể score nhưng phải phát cảnh báo/quality metric.

## 6. Train/serve parity checklist

- [x] `FEATURE_COLUMNS` là thứ tự canonical dùng khi train, batch và API.
- [x] Serialized bundle chứa model + metadata + schema/version/hash.
- [x] Batch và API gọi cùng `score_frame`.
- [x] API cấm extra fields, range sai và mâu thuẫn phone/internet.
- [x] Unit test giữ tie-break top-K, schema, checksum và manifest.
- [ ] Với dữ liệu thật: thêm point-in-time feature query, feature freshness SLA và contract test với warehouse.

Chạy benchmark đã khóa bằng `python -m telco_churn.train`; notebook này không thay thế benchmark report hay MLflow run.